In [ ]:
from netCDF4 import Dataset
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import datetime
from tqdm import tqdm
from glob import glob
import pickle
import random
import os
import fnmatch

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset as TorchDataset
import torch.nn.functional as F
from torchvision import datasets, transforms
from typing import Tuple, List, Type, Dict, Any
from torch.utils.tensorboard import SummaryWriter

from SGDR import CosineAnnealingWarmRestarts
from mish import Mish
from MyResidualNetwork import MyResNet, MyBasicBlock
from MyDataPreparationMix_norm import CustomDataset
from autoencoder import Encoder, Decoder

In [ ]:
torch.cuda.empty_cache()

In [ ]:
def drawing(tensor1, tensor2, indexes):
    for i in indexes:
        extracted_tensor1 = tensor1[i, 0, :, :]
        extracted_tensor2 = tensor2[i, 0, :, :]

        array1 = extracted_tensor1.detach().cpu().numpy()
        array2 = extracted_tensor2.detach().cpu().numpy()
        
        vmin = min(array1.min(), array2.min())
        vmax = max(array1.max(), array2.max())

        fig, axs = plt.subplots(1, 2, figsize=(12, 8))

        cax1 = axs[0].imshow(array1, cmap='viridis', aspect='auto', vmin=vmin, vmax=vmax)
        axs[0].set_title('Sample SSS')

        cax2 = axs[1].imshow(array2, cmap='viridis', aspect='auto', vmin=vmin, vmax=vmax)
        axs[1].set_title('Decoded SSS')

        cbar = fig.colorbar(cax1, ax=axs, orientation='vertical', fraction=0.02, pad=0.04)
        cbar.ax.set_ylabel('sss')

        plt.tight_layout()
        plt.show()
        
        print('--------------------------------------------------------------------------------------------------')

In [ ]:
run_name = 'sss_pre_autoencoder_run004'

In [ ]:
device = torch.device('cuda:1')

In [ ]:
encoder = torch.load(f'/app/Kara_plume_movement/sss/models/model_{run_name}_encoder.pth', map_location=torch.device('cpu'));
decoder = torch.load(f'/app/Kara_plume_movement/sss/models/model_{run_name}_decoder.pth', map_location=torch.device('cpu'));

In [ ]:
encoder.eval();
decoder.eval();

In [ ]:
encoder = encoder.cuda()
decoder = decoder.cuda()

In [ ]:
dates1 = {
    '2010': {'start': '0711', 'finish': '1009'},
    '2011': {'start': '0625', 'finish': '1029'},
    '2012': {'start': '0701', 'finish': '1105'},
    '2013': {'start': '0713', 'finish': '1018'},
    '2014': {'start': '0716', 'finish': '1023'},
    '2015': {'start': '0630', 'finish': '1025'},
    '2016': {'start': '0709', 'finish': '1031'},
    '2017': {'start': '0715', 'finish': '1020'},
    '2018': {'start': '0801', 'finish': '1031'},
    '2019': {'start': '0710', 'finish': '1025'},
    '2020': {'start': '0707', 'finish': '1031'},
    '2021': {'start': '0710', 'finish': '1025'},
    '2022': {'start': '0701', 'finish': '1031'},
    '2023': {'start': '0720', 'finish': '1010'},
}

dates2 = {
    '2015': {'start': '0630', 'finish': '1025'},
    '2016': {'start': '0709', 'finish': '1031'},
    '2017': {'start': '0715', 'finish': '1020'},
    '2018': {'start': '0801', 'finish': '1031'},
    '2019': {'start': '0710', 'finish': '1025'},
    '2020': {'start': '0707', 'finish': '1031'},
    '2021': {'start': '0710', 'finish': '1025'},
    '2022': {'start': '0701', 'finish': '1031'},
    '2023': {'start': '0720', 'finish': '1010'},
}

In [ ]:
batch_size = 16

In [ ]:
dataset = CustomDataset(dates_dict1=dates1, path1='/mnt/hippocamp/asavin/data/ESACCI/ESACCI_norm',
                        dates_dict2=dates2, path2='/mnt/hippocamp/asavin/data/SSS_ESACCI_grid/SSS_ESACCI_grid_norm_free_ice_dates')
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
loss_function=torch.nn.MSELoss()

In [ ]:
data, mask = next(iter(dataloader))
data.shape, mask.shape

In [ ]:
data_gpu = data.to(device='cuda', dtype=torch.float)
mask_gpu = mask.to(device='cuda', dtype=torch.float)

encoded_data = encoder.forward(data_gpu)
decoded_data = decoder.forward(encoded_data)

data_gpu_masked = data_gpu[mask_gpu == 1]
result_masked = decoded_data[mask_gpu == 1]

loss = loss_function(data_gpu_masked, result_masked)
test_loss = loss.detach()

In [ ]:
encoded_data.shape, decoded_data.shape

In [ ]:
loss

In [ ]:
decoded_data_m = decoded_data * mask_gpu
data_gpu_m = data_gpu * mask_gpu
decoded_data_m.shape, data_gpu.shape

In [ ]:
data_gpu_m.mean(), data_gpu_m.std(), decoded_data_m.mean(), decoded_data_m.std()

In [ ]:
drawing(data_gpu_m, decoded_data_m, [i for i in range(data_gpu.shape[0])])